In [0]:
%sql
USE CATALOG corretora;

CREATE TABLE IF NOT EXISTS ops.controle_silver (
  fluxo                 STRING NOT NULL,
  tabela_silver         STRING,
  ultimo_ingerido_em    TIMESTAMP COMMENT 'Watermark: maior _data_ingestao da Bronze já processado',
  registros_lidos       BIGINT,
  registros_validos     BIGINT,
  registros_quarentena  BIGINT,
  atualizado_em         TIMESTAMP
) COMMENT 'Watermark e métricas da última execução de cada fluxo Bronze → Silver';

CREATE TABLE IF NOT EXISTS silver.quarentena (
  id_registro      STRING NOT NULL COMMENT 'Hash do registro: a mesma rejeição nunca entra duas vezes',
  tabela_destino   STRING,
  fluxo            STRING,
  regras_violadas  ARRAY<STRING>,
  registro         STRING COMMENT 'Registro rejeitado, em JSON',
  quarentenado_em  TIMESTAMP
) COMMENT 'Registros que não passaram nas regras de qualidade da Silver';

In [0]:
%sql
USE CATALOG corretora;

CREATE TABLE IF NOT EXISTS silver.cotacoes_intradiarias (
  ativo             STRING        NOT NULL,
  fonte             STRING        NOT NULL,
  cotado_em_utc     TIMESTAMP     NOT NULL COMMENT 'Horário da cotação informado pela fonte',
  moeda_cotacao     STRING,
  preco_compra      DECIMAL(20,8) COMMENT 'Bid: preço pelo qual o mercado compra',
  preco_venda       DECIMAL(20,8) COMMENT 'Ask: preço pelo qual o mercado vende',
  preco_maximo_dia  DECIMAL(20,8),
  preco_minimo_dia  DECIMAL(20,8),
  variacao_pct_dia  DECIMAL(12,4),
  coletado_em_utc   TIMESTAMP     COMMENT 'Quando a plataforma coletou a cotação',
  _id_coleta        STRING,
  _arquivo_origem   STRING,
  _processado_em    TIMESTAMP
) COMMENT 'Cotações intradiárias de câmbio, cripto e B3 em modelo canônico'
TBLPROPERTIES (delta.enableChangeDataFeed = true);

CREATE TABLE IF NOT EXISTS silver.precos_diarios (
  ativo             STRING        NOT NULL,
  fonte             STRING        NOT NULL,
  data_pregao       DATE          NOT NULL COMMENT 'Data no horário de Brasília',
  preco_abertura    DECIMAL(20,8),
  preco_maximo      DECIMAL(20,8),
  preco_minimo      DECIMAL(20,8),
  preco_fechamento  DECIMAL(20,8),
  volume            BIGINT,
  cotado_em_utc     TIMESTAMP,
  coletado_em_utc   TIMESTAMP,
  _id_coleta        STRING,
  _arquivo_origem   STRING,
  _processado_em    TIMESTAMP
) COMMENT 'Preços diários (OHLC) por ativo'
TBLPROPERTIES (delta.enableChangeDataFeed = true);

CREATE TABLE IF NOT EXISTS silver.ativos (
  ativo             STRING NOT NULL,
  nome              STRING,
  classe            STRING COMMENT 'MOEDA, CRIPTO, ACAO, FUNDO, BDR, INDICE ou OUTRO',
  setor             STRING,
  moeda_cotacao     STRING,
  casas_decimais    INT,
  negociavel        BOOLEAN COMMENT 'Índices não são negociáveis',
  fonte             STRING,
  coletado_em_utc   TIMESTAMP,
  _id_coleta        STRING,
  _arquivo_origem   STRING,
  _processado_em    TIMESTAMP
) COMMENT 'Catálogo unificado de ativos'
TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql
USE CATALOG corretora;

CREATE TABLE IF NOT EXISTS silver.ref_series_bcb (
  codigo_sgs     INT NOT NULL,
  nome_serie     STRING,
  unidade        STRING,
  periodicidade  STRING
) COMMENT 'Referência das séries do SGS usadas pela corretora';

INSERT OVERWRITE silver.ref_series_bcb VALUES
  (432, 'Meta Selic (Copom)', '% a.a.', 'DIARIA'),
  (11,  'Selic diária',       '% a.d.', 'DIARIA'),
  (12,  'CDI diário',         '% a.d.', 'DIARIA'),
  (433, 'IPCA',               '% a.m.', 'MENSAL'),
  (189, 'IGP-M',              '% a.m.', 'MENSAL'),
  (1,   'Dólar PTAX venda',   'R$/US$', 'DIARIA');

CREATE TABLE IF NOT EXISTS silver.indicadores_economicos (
  codigo_sgs       INT  NOT NULL,
  data_referencia  DATE NOT NULL,
  nome_serie       STRING,
  unidade          STRING,
  periodicidade    STRING,
  valor            DECIMAL(20,8),
  coletado_em_utc  TIMESTAMP,
  _id_coleta       STRING,
  _arquivo_origem  STRING,
  _processado_em   TIMESTAMP
) COMMENT 'Indicadores do Banco Central com nome e unidade'
TBLPROPERTIES (delta.enableChangeDataFeed = true);

CREATE TABLE IF NOT EXISTS silver.tesouro_precos_taxas (
  tipo_titulo      STRING NOT NULL,
  data_vencimento  DATE   NOT NULL,
  data_base        DATE   NOT NULL,
  indexador        STRING COMMENT 'SELIC, PREFIXADO, IPCA ou IGPM',
  taxa_compra      DECIMAL(10,4),
  taxa_venda       DECIMAL(10,4),
  pu_compra        DECIMAL(18,6),
  pu_venda         DECIMAL(18,6),
  pu_base          DECIMAL(18,6),
  _ingerido_em     TIMESTAMP,
  _arquivo_origem  STRING,
  _processado_em   TIMESTAMP
) COMMENT 'Preços e taxas dos títulos do Tesouro Direto'
TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql
USE CATALOG corretora;

CREATE TABLE IF NOT EXISTS silver.clientes (
  id_cliente            STRING NOT NULL,
  nome                  STRING,
  cpf                   STRING COMMENT 'Somente dígitos. Dado pessoal (LGPD)',
  email                 STRING COMMENT 'Dado pessoal (LGPD)',
  data_nascimento       DATE,
  cidade                STRING,
  uf                    STRING,
  perfil_investidor     STRING,
  faixa_renda           STRING,
  cadastrado_em_utc     TIMESTAMP,
  ultimo_evento_em_utc  TIMESTAMP,
  _arquivo_origem       STRING,
  _processado_em        TIMESTAMP
) COMMENT 'Estado atual de cada cliente, com atualizações parciais aplicadas'
TBLPROPERTIES (delta.enableChangeDataFeed = true);

CREATE TABLE IF NOT EXISTS silver.operacoes (
  id_operacao       STRING NOT NULL,
  id_cliente        STRING NOT NULL,
  ativo             STRING NOT NULL,
  classe_ativo      STRING,
  lado              STRING,
  quantidade        DECIMAL(28,8),
  preco_mercado     DECIMAL(20,8),
  preco_executado   DECIMAL(20,8),
  valor_financeiro  DECIMAL(18,2),
  taxa_corretagem   DECIMAL(18,2),
  receita_spread    DECIMAL(18,2),
  canal             STRING,
  status            STRING,
  executada_em_utc  TIMESTAMP,
  data_operacao     DATE COMMENT 'Data no horário de Brasília',
  _ingerido_em      TIMESTAMP,
  _arquivo_origem   STRING,
  _processado_em    TIMESTAMP
) COMMENT 'Operações executadas, tipadas e validadas'
TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql
USE CATALOG corretora;

ALTER TABLE silver.cotacoes_intradiarias DROP CONSTRAINT IF EXISTS preco_positivo;
ALTER TABLE silver.cotacoes_intradiarias ADD CONSTRAINT preco_positivo CHECK (preco_compra > 0);
ALTER TABLE silver.cotacoes_intradiarias DROP CONSTRAINT IF EXISTS spread_valido;
ALTER TABLE silver.cotacoes_intradiarias ADD CONSTRAINT spread_valido CHECK (preco_venda >= preco_compra);

ALTER TABLE silver.precos_diarios DROP CONSTRAINT IF EXISTS fechamento_positivo;
ALTER TABLE silver.precos_diarios ADD CONSTRAINT fechamento_positivo CHECK (preco_fechamento > 0);

ALTER TABLE silver.operacoes DROP CONSTRAINT IF EXISTS quantidade_positiva;
ALTER TABLE silver.operacoes ADD CONSTRAINT quantidade_positiva CHECK (quantidade > 0);
ALTER TABLE silver.operacoes DROP CONSTRAINT IF EXISTS lado_valido;
ALTER TABLE silver.operacoes ADD CONSTRAINT lado_valido CHECK (lado IN ('COMPRA', 'VENDA'));